# 🔥 DWSIM - Optimize Natural Gas Processing

> **Created by:**
> Prof. Roymel Rodríguez Carpio, Ph.D.
> Prof. Nicolas Spogis, Ph.D.
> Gabriel Freitas

---

## Import the Libraries

### Libraries for array and dataset manipulation

In [1]:
import numpy as np
import pandas as pd
import time

### Libraries for plotting results

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns


### Set NumPy Seed

In [3]:
seed = 42
np.random.seed(seed)

## Import the Specific Libraries to run DWSIM

In [4]:
# remove the following two lines to run on linux
import pythoncom  # Component of the pywin32 library used to interact with COM (Component Object Model)

pythoncom.CoInitialize()

In [5]:
import clr  # Pythonnet

## Change your DWSIM installation path and load dlls

In [6]:
dwsimpath = r"C:\Users\nicol\AppData\Local\DWSIM"

In [7]:
clr.AddReference(dwsimpath + "\\CapeOpen.dll")
clr.AddReference(dwsimpath + "\\DWSIM.Automation.dll")
clr.AddReference(dwsimpath + "\\DWSIM.Interfaces.dll")
clr.AddReference(dwsimpath + "\\DWSIM.GlobalSettings.dll")
clr.AddReference(dwsimpath + "\\DWSIM.SharedClasses.dll")
clr.AddReference(dwsimpath + "\\DWSIM.Thermodynamics.dll")
clr.AddReference(dwsimpath + "\\DWSIM.UnitOperations.dll")
clr.AddReference(dwsimpath + "\\DWSIM.Inspector.dll")
clr.AddReference(dwsimpath + "\\System.Buffers.dll")
clr.AddReference(dwsimpath + "\\DWSIM.Thermodynamics.ThermoC.dll")



## Open DWSIM Automation def

In [8]:
def open_DWSIM(FlowsheetFile):
    from DWSIM.Automation import Automation3
    Manager = Automation3()
    Flowsheet = Manager.LoadFlowsheet(FlowsheetFile)
    return Manager, Flowsheet

## Read and Save Snapshot defs

In [9]:
def SaveSnapshot(dwsimpath, Flowsheet):
    clr.AddReference(dwsimpath + "\\DWSIM.Interfaces.dll")
    from DWSIM.Interfaces.Enums import SnapshotType
    Type = SnapshotType.ObjectData
    snap = Flowsheet.GetSnapshot(Type)
    return snap


def ReadSnapshot(dwsimpath, snap, Flowsheet):
    clr.AddReference(dwsimpath + "\\DWSIM.Interfaces.dll")
    from DWSIM.Interfaces.Enums import SnapshotType
    Type = SnapshotType.ObjectData
    Flowsheet.RestoreSnapshot(snap, Type)
    return snap

## Open DWSIM File and load initial Snapshot

In [10]:
FlowsheetFileName = "DWSIM/NaturalGasProcessing.dwxmz"
Manager, Flowsheet = open_DWSIM(FlowsheetFileName)

In [11]:
Snapshot = SaveSnapshot(dwsimpath, Flowsheet)

## Call DWSIM -> Change Input Parameters -> Solve -> Get Output Parameters

In [12]:
def run_dwsim_simulation(u):
    global Manager, Flowsheet, Snapshot, dwsimpath

    # Read Snapshot
    ReadSnapshot(dwsimpath, Snapshot, Flowsheet)

    # Extract decision variables
    V_123102_Temp = u[0]
    T_123701_Reboiler = u[1]
    T_123702_Condenser = u[2]
    T_123702_Reboiler = u[3]
    T_123703_Reboiler = u[4]

    # Set Spreadsheet Object
    mySpreadsheet = Flowsheet.GetSpreadsheetObject()
    ws = mySpreadsheet.Worksheets["MAIN"]

    try:
        # Set Input Parameters
        ws.Cells["B2"].Data = V_123102_Temp
        ws.Cells["B3"].Data = T_123701_Reboiler
        ws.Cells["B4"].Data = T_123702_Condenser
        ws.Cells["B5"].Data = T_123702_Reboiler
        ws.Cells["B6"].Data = T_123703_Reboiler

        # Update Input Spreadsheet
        ws.Recalculate()

        # Request a calculation
        Manager.CalculateFlowsheet4(Flowsheet)

        # Check if the Flowsheet was Calculated Correctly
        if Flowsheet.Solved == False:
            # Penalty for failed simulation
            return np.array([1e6, 1e6, 1e6, 1e6, 1e6,
                         1e6, 1e6, 1e6, 1e6, 1e6, 1e6])

        # Update Output Spreadsheet
        ws.Recalculate()

        # Get Output Parameters
        Sale_Price = float(ws.Cells["B17"].Data)
        GVC1 = float(ws.Cells["B10"].Data)
        GVC2 = float(ws.Cells["B12"].Data)
        GVC5 = float(ws.Cells["B15"].Data)
        GVC5_PVR = float(ws.Cells["B16"].Data)

        # Get Auxiliary Output Parameters
        T_123701_Stage1_Temperature = float(ws.Cells["B19"].Data)
        T_123701_Stage11_Temperature = float(ws.Cells["B20"].Data)
        T_123702_Stage1_Temperature = float(ws.Cells["B21"].Data)
        T_123702_Stage21_Temperature = float(ws.Cells["B22"].Data)
        T_123703_Stage1_Temperature = float(ws.Cells["B23"].Data)
        T_123703_Stage6_Temperature = float(ws.Cells["B24"].Data)

        # ========================================
        # Objective Function: Maximize Sales Price
        # ========================================
        f_obj = -Sale_Price  # we want to maximize this

        # ========================================
        # Constraints
        # ========================================
        g1 = GVC1  # Should be >= 80 %
        g2 = GVC2  # Should be <=12 %
        g3 = GVC5  # Should be <=2 %
        g4 = GVC5_PVR  # Should be <= 76 kPa

        return np.array([f_obj, g1, g2, g3, g4,
                         T_123701_Stage1_Temperature,
                         T_123701_Stage11_Temperature,
                         T_123702_Stage1_Temperature,
                         T_123702_Stage21_Temperature,
                         T_123703_Stage1_Temperature,
                         T_123703_Stage6_Temperature,])

    except Exception as e:
        print(f"Error in simulation: {e}")
        return np.array([1e6, 1e6, 1e6, 1e6, 1e6,
                         1e6, 1e6, 1e6, 1e6, 1e6, 1e6])



## Variable bounds

In [13]:
# Decision variable bounds
# [V_123102_Temp, T_123701_Reboiler, T_123702_Condenser, T_123702_Reboiler, T_123703_Reboiler]
LB = np.array([-33.0, 60.0, 0.7,  9.0, 45.0])  # Lower bounds
UB = np.array([-17.0, 77.0, 4.5, 27.0, 64.0])  # Upper bounds

bounds = list(zip(LB, UB))

print("\nDecision Variable Bounds:")
print(f"  V_123102_Temp: [{LB[0]:.0f}, {UB[0]:.0f}]")
print(f"  T_123701_Reboiler: [{LB[1]:.0f}, {UB[1]:.0f}]")
print(f"  T_123702_Condenser: [{LB[2]:.0f}, {UB[2]:.0f}]")
print(f"  T_123702_Reboiler: [{LB[3]:.0f}, {UB[3]:.0f}]")
print(f"  T_123703_Reboiler: [{LB[4]:.0f}, {UB[4]:.0f}]")



Decision Variable Bounds:
  V_123102_Temp: [-33, -17]
  T_123701_Reboiler: [60, 77]
  T_123702_Condenser: [1, 4]
  T_123702_Reboiler: [9, 27]
  T_123703_Reboiler: [45, 64]


## Test simulation with initial values

In [14]:
print("=" * 60)
print("TESTING DWSIM SIMULATION - Lower bounds")
print("=" * 60)

(f_obj, g1, g2, g3, g4,
 T_123701_Stage1_Temperature,
 T_123701_Stage11_Temperature,
 T_123702_Stage1_Temperature,
 T_123702_Stage21_Temperature,
 T_123703_Stage1_Temperature,
 T_123703_Stage6_Temperature) = run_dwsim_simulation(LB)

print(f"\nTest Input:")
print(f"  V_123102_Temp = {LB[0]:.2f}")
print(f"  T_123701_Reboiler = {LB[1]:.2f}")
print(f"  T_123702_Condenser = {LB[2]:.2f}")
print(f"  T_123702_Reboiler = {LB[3]:.2f}")
print(f"  T_123703_Reboiler = {LB[4]:.2f}")

print(f"\nTest Output:")
print(f"  Objective (Sales Price) = {-f_obj:.2f} US$")
print(f"  g1 (GV.C1 [%]) = {g1:.2f} (should be >= 80 %)")
print(f"  g2 (GLP.C2 [%]) = {g2:.2f} (should be <=12 %)")
print(f"  g3 (GLP.C5 [%]) = {g3:.2f} (should be <=2 %)")
print(f"  g4 (C5+.PVR [kPa]) = {g4:.2f} (should be <= 76 kPa)")

print(f"\nTemperatures Output:")
print(f"  T 123701 Stage 1 Temperature = {T_123701_Stage1_Temperature:.2f} ºC")
print(f"  T 123701 Stage 11 Temperature = {T_123701_Stage11_Temperature:.2f} ºC")
print(f"  T 123702 Stage 1 Temperature = {T_123702_Stage1_Temperature:.2f} ºC")
print(f"  T 123702 Stage 21 Temperature = {T_123702_Stage21_Temperature:.2f} ºC")
print(f"  T 123703 Stage 1 Temperature = {T_123703_Stage1_Temperature:.2f} ºC")
print(f"  T 123703 Stage 6 Temperature = {T_123703_Stage6_Temperature:.2f} ºC")

TESTING DWSIM SIMULATION - Lower bounds

Test Input:
  V_123102_Temp = -33.00
  T_123701_Reboiler = 60.00
  T_123702_Condenser = 0.70
  T_123702_Reboiler = 9.00
  T_123703_Reboiler = 45.00

Test Output:
  Objective (Sales Price) = 658.72 US$
  g1 (GV.C1 [%]) = 83.54 (should be >= 80 %)
  g2 (GLP.C2 [%]) = 8.08 (should be <=12 %)
  g3 (GLP.C5 [%]) = 6.71 (should be <=2 %)
  g4 (C5+.PVR [kPa]) = 32.65 (should be <= 76 kPa)

Temperatures Output:
  T 123701 Stage 1 Temperature = 16.89 ºC
  T 123701 Stage 11 Temperature = 71.36 ºC
  T 123702 Stage 1 Temperature = 42.77 ºC
  T 123702 Stage 21 Temperature = 166.33 ºC
  T 123703 Stage 1 Temperature = 111.86 ºC
  T 123703 Stage 6 Temperature = 205.59 ºC


In [15]:
print("=" * 60)
print("TESTING DWSIM SIMULATION - Upper bounds")
print("=" * 60)

(f_obj, g1, g2, g3, g4,
 T_123701_Stage1_Temperature,
 T_123701_Stage11_Temperature,
 T_123702_Stage1_Temperature,
 T_123702_Stage21_Temperature,
 T_123703_Stage1_Temperature,
 T_123703_Stage6_Temperature) = run_dwsim_simulation(UB)


print(f"\nTest Input:")
print(f"  V_123102_Temp = {UB[0]:.2f}")
print(f"  T_123701_Reboiler = {UB[1]:.2f}")
print(f"  T_123702_Condenser = {UB[2]:.2f}")
print(f"  T_123702_Reboiler = {UB[3]:.2f}")
print(f"  T_123703_Reboiler = {UB[4]:.2f}")

print(f"\nTest Output:")
print(f"  Objective (Sales Price) = {-f_obj:.2f} US$")
print(f"  g1 (GV.C1 [%]) = {g1:.2f} (should be >= 80 %)")
print(f"  g2 (GLP.C2 [%]) = {g2:.2f} (should be <=12 %)")
print(f"  g3 (GLP.C5 [%]) = {g3:.2f} (should be <=2 %)")
print(f"  g4 (C5+.PVR [kPa]) = {g4:.2f} (should be <= 76 kPa)")

print(f"\nTemperatures Output:")
print(f"  T 123701 Stage 1 Temperature = {T_123701_Stage1_Temperature:.2f} ºC")
print(f"  T 123701 Stage 11 Temperature = {T_123701_Stage11_Temperature:.2f} ºC")
print(f"  T 123702 Stage 1 Temperature = {T_123702_Stage1_Temperature:.2f} ºC")
print(f"  T 123702 Stage 21 Temperature = {T_123702_Stage21_Temperature:.2f} ºC")
print(f"  T 123703 Stage 1 Temperature = {T_123703_Stage1_Temperature:.2f} ºC")
print(f"  T 123703 Stage 6 Temperature = {T_123703_Stage6_Temperature:.2f} ºC")

TESTING DWSIM SIMULATION - Upper bounds

Test Input:
  V_123102_Temp = -17.00
  T_123701_Reboiler = 77.00
  T_123702_Condenser = 4.50
  T_123702_Reboiler = 27.00
  T_123703_Reboiler = 64.00

Test Output:
  Objective (Sales Price) = 626.16 US$
  g1 (GV.C1 [%]) = 81.13 (should be >= 80 %)
  g2 (GLP.C2 [%]) = 14.28 (should be <=12 %)
  g3 (GLP.C5 [%]) = 0.00 (should be <=2 %)
  g4 (C5+.PVR [kPa]) = 117.01 (should be <= 76 kPa)

Temperatures Output:
  T 123701 Stage 1 Temperature = 14.40 ºC
  T 123701 Stage 11 Temperature = 69.11 ºC
  T 123702 Stage 1 Temperature = 30.74 ºC
  T 123702 Stage 21 Temperature = 120.65 ºC
  T 123703 Stage 1 Temperature = 91.23 ºC
  T 123703 Stage 6 Temperature = 151.74 ºC


## Class Evaluator to avoid recomputation

In [16]:
class Evaluator:
    def __init__(self):
        self.cache = {}
        self.n_evaluations = 0

    def evaluate(self, x):
        key = tuple(np.round(x, 6))  # Less precision for caching
        if key not in self.cache:
            self.cache[key] = run_dwsim_simulation(x)
            self.n_evaluations += 1
        return self.cache[key]

    def objective(self, x):
        return self.evaluate(x)[0]

    def g1(self, x):
        return self.evaluate(x)[1]

    def g2(self, x):
        return self.evaluate(x)[2]

    def g3(self, x):
        return self.evaluate(x)[3]

    def g4(self, x):
        return self.evaluate(x)[4]

    def reset_counter(self):
        self.n_evaluations = 0

## Create global evaluator

In [17]:
ev = Evaluator()

## Constraint checking function

In [18]:
def check_constraints_scipy(x, constraints, tol=1e-6):
    """Check constraints for scipy optimization"""
    report = []

    for i, c in enumerate(constraints, 1):
        val = np.atleast_1d(c.fun(x))
        lb = np.atleast_1d(c.lb)
        ub = np.atleast_1d(c.ub)

        viol_low = np.maximum(lb - val, 0.0)
        viol_high = np.maximum(val - ub, 0.0)
        violation = np.maximum(viol_low, viol_high)

        ok = np.all(violation <= tol)

        report.append({
            "id": i,
            "ok": ok,
            "value": val,
            "lb": lb,
            "ub": ub,
            "violation": violation
        })

    return report

# ========================================
# Create LHS Dataset
# ========================================

In [19]:
from scipy.stats import qmc, norm
from tqdm import tqdm

In [20]:
# Bounds array: each row is [min, max] for each variable
LHS_bounds = np.column_stack((LB, UB))

In [21]:
n_samples = 2000
n_variables = LHS_bounds.shape[0]

In [22]:
# =============================================================================
# Generate LHS samples
# =============================================================================
print("Generating Latin Hypercube Samples...")

# Create LHS sampler with seed for reproducibility
sampler = qmc.LatinHypercube(d=n_variables, seed=42)

# Generate samples in [0, 1] range
lhs_samples_unit = sampler.random(n=n_samples)

# Scale samples to actual bounds
lhs_samples = qmc.scale(lhs_samples_unit, LHS_bounds[:, 0], LHS_bounds[:, 1])

print(f"Generated {n_samples} LHS samples successfully!")

Generating Latin Hypercube Samples...
Generated 2000 LHS samples successfully!


In [23]:
# =============================================================================
# Run DWSIM simulations for all samples
# =============================================================================
print(f"\nRunning DWSIM simulations for {n_samples} samples...")
print("This may take a while...\n")

# Initialize output arrays
outputs = np.zeros((n_samples, 11))

# Run simulations with progress bar
for i, sample in enumerate(tqdm(lhs_samples, desc="Simulating", unit="sample")):
    outputs[i, :] = run_dwsim_simulation(sample)

print("\nSimulations completed!")


Running DWSIM simulations for 2000 samples...
This may take a while...



Simulating: 100%|██████████| 2000/2000 [2:08:20<00:00,  3.85s/sample]  


Simulations completed!


In [24]:
# =============================================================================
# Create DataFrame and save dataset
# =============================================================================
# Column names
input_columns = ['V_123102_Temp',
                 'T_123701_Reboiler',
                 'T_123702_Condenser',
                 'T_123702_Reboiler',
                 'T_123703_Reboiler']

output_columns = ['Sales_Price', 'GVC1', 'GVC2', 'GVC5', 'GVC5_PVR',
                  'T_123701_Stage1_Temperature',
                  'T_123701_Stage11_Temperature',
                  'T_123702_Stage1_Temperature',
                  'T_123702_Stage21_Temperature',
                  'T_123703_Stage1_Temperature',
                  'T_123703_Stage6_Temperature']

# Create DataFrame
df = pd.DataFrame(
    data=np.hstack([lhs_samples, outputs]),
    columns=input_columns + output_columns
)

# Count valid and invalid samples
mask_invalid = (df[output_columns] == 1e6).all(axis=1)
df_clean = df[~mask_invalid]
n_valid = df_clean.shape[0]
n_invalid = n_samples - n_valid

print(f"\n{'='*60}")
print(f"Dataset Summary")
print(f"{'='*60}")
print(f"Total samples:   {n_samples}")
print(f"Valid samples:   {n_valid} ({100*n_valid/n_samples:.1f}%)")
print(f"Invalid samples: {n_invalid} ({100*n_invalid/n_samples:.1f}%)")
print(f"{'='*60}")


Dataset Summary
Total samples:   2000
Valid samples:   2000 (100.0%)
Invalid samples: 0 (0.0%)


In [25]:
# Display statistics
print("\nDataset Statistics:")
print(df.describe().round(4))


Dataset Statistics:
       V_123102_Temp  T_123701_Reboiler  T_123702_Condenser  \
count      2000.0000          2000.0000           2000.0000   
mean        -24.9999            68.5000              2.6000   
std           4.6200             4.9087              1.0973   
min         -33.0000            60.0038              0.7011   
25%         -28.9971            64.2485              1.6501   
50%         -24.9984            68.5027              2.6007   
75%         -21.0046            72.7450              3.5500   
max         -17.0031            76.9983              4.4994   

       T_123702_Reboiler  T_123703_Reboiler  Sales_Price       GVC1  \
count          2000.0000          2000.0000    2000.0000  2000.0000   
mean             18.0000            54.5000    -644.5029    82.4302   
std               5.1974             5.4862      13.8967     0.9233   
min               9.0069            45.0044    -673.3515    80.7304   
25%              13.5050            49.7551    -656.4881

In [26]:
# =============================================================================
# Save dataset to CSV
# =============================================================================
# Save full dataset
output_file_full = "datasets/dataset_full.csv"
df.to_csv(output_file_full, index=False)
print(f"\nFull dataset saved to: {output_file_full}")

# Save clean dataset
output_file_clean = "datasets/dataset_clean.csv"
df_clean.to_csv(output_file_clean, index=False)
print(f"Clean dataset saved to: {output_file_clean}")

print(f"\n{'='*60}")
print("Dataset generation completed successfully!")
print(f"{'='*60}")


Full dataset saved to: datasets/dataset_full.csv
Clean dataset saved to: datasets/dataset_clean.csv

Dataset generation completed successfully!


# ========================================
# LOCAL OPTIMIZATION: SLSQP (Scipy - Baseline)
# ========================================

In [27]:
# ============================================================
# Load LHS dataset
# ============================================================
df = pd.read_csv("datasets/dataset_clean.csv")

# Column names
decision_vars = [
    'V_123102_Temp',
    'T_123701_Reboiler',
    'T_123702_Condenser',
    'T_123702_Reboiler',
    'T_123703_Reboiler',]

obj_col = 'Sales_Price'
g1_col = 'GVC1'       # >= 80
g2_col = 'GVC2'       # <= 12
g3_col = 'GVC5'       # <= 2
g4_col = 'GVC5_PVR'   # <= 76

# ============================================================
# Define constraint limits
# ============================================================
constraints_limits = {
    g1_col: {'lb': 80.0, 'ub': np.inf},     # g1 >= 80
    g2_col: {'lb': 0.0, 'ub': 12.0},        # g2 <= 12
    g3_col: {'lb': 0.0, 'ub': 2.0},         # g3 <= 2
    g4_col: {'lb': 0.0, 'ub': 76.0},        # g4 <= 76
}

# ============================================================
# Check feasibility
# ============================================================
feasible_mask = pd.Series(True, index=df.index)

for col, lim in constraints_limits.items():
    if np.isfinite(lim['lb']):
        feasible_mask &= (df[col] >= lim['lb'])
    if np.isfinite(lim['ub']):
        feasible_mask &= (df[col] <= lim['ub'])

df_feasible = df[feasible_mask]
n_total = len(df)
n_feasible = len(df_feasible)

print(f"Total LHS samples: {n_total}")
print(f"Feasible samples:  {n_feasible} ({100*n_feasible/n_total:.1f}%)")

# ============================================================
# Select best x0
# ============================================================
if n_feasible > 0:
    # Best feasible point (min Sales_Price = most negative = highest real price)
    best_idx = df_feasible[obj_col].idxmin()
    best_row = df_feasible.loc[best_idx]
    print(f"\n--- Best feasible point (index {best_idx}) ---")
else:
    # No feasible point: pick the one with minimum total violation
    print("\nNo fully feasible point found. Selecting least-violated point...")

    violation = pd.Series(0.0, index=df.index)
    for col, lim in constraints_limits.items():
        if np.isfinite(lim['lb']):
            violation += np.maximum(lim['lb'] - df[col], 0.0)
        if np.isfinite(lim['ub']):
            violation += np.maximum(df[col] - lim['ub'], 0.0)

    best_idx = violation.idxmin()
    best_row = df.loc[best_idx]
    print(f"\n--- Least-violated point (index {best_idx}, "
          f"total violation = {violation[best_idx]:.4f}) ---")

# Extract x0
x0 = best_row[decision_vars].values.astype(float)

print(f"\nObjective (Sales_Price): {best_row[obj_col]:.4f}")
print(f"Real Price: {-best_row[obj_col]:.4f}")
print(f"\nDecision variables (x0):")
for name, val in zip(decision_vars, x0):
    print(f"  {name} = {val:.4f}")

print(f"\nConstraint values:")
for col, lim in constraints_limits.items():
    val = best_row[col]
    lb_str = f"{lim['lb']:.1f}" if np.isfinite(lim['lb']) else "-inf"
    ub_str = f"{lim['ub']:.1f}" if np.isfinite(lim['ub']) else "inf"
    ok = True
    if np.isfinite(lim['lb']) and val < lim['lb']:
        ok = False
    if np.isfinite(lim['ub']) and val > lim['ub']:
        ok = False
    status = "✓" if ok else "✗"
    print(f"  {col} = {val:.4f}  [{lb_str}, {ub_str}]  {status}")

# ============================================================
# x0 is ready to use in your optimization
# ============================================================
print(f"\nx0 = np.array({list(x0)})")

Total LHS samples: 2000
Feasible samples:  281 (14.1%)

--- Best feasible point (index 1591) ---

Objective (Sales_Price): -662.8284
Real Price: 662.8284

Decision variables (x0):
  V_123102_Temp = -32.8294
  T_123701_Reboiler = 64.0769
  T_123702_Condenser = 4.2981
  T_123702_Reboiler = 13.8861
  T_123703_Reboiler = 47.4421

Constraint values:
  GVC1 = 83.7331  [80.0, inf]  ✓
  GVC2 = 11.9135  [0.0, 12.0]  ✓
  GVC5 = 0.1878  [0.0, 2.0]  ✓
  GVC5_PVR = 53.5710  [0.0, 76.0]  ✓

x0 = np.array([-32.82942291491444, 64.07694006124437, 4.298112932449205, 13.886056746901064, 47.442059795382654])


In [28]:
from scipy.optimize import minimize, NonlinearConstraint

print("\n\n" + "=" * 60)
print("OPTIMIZATION 1: SLSQP (SciPy - Local Optimizer)")
print("=" * 60)

# Reset evaluator
ev.reset_counter()
ev.cache.clear()

# Constraints for SLSQP
constraints_scipy = [
    NonlinearConstraint(ev.g1, 80.0, np.inf),       # should be >= 80
    NonlinearConstraint(ev.g2, 0.0, 12.0),          # should be <=12 %
    NonlinearConstraint(ev.g3, 0.0, 2.0),           # should be <=2 %
    NonlinearConstraint(ev.g4, 0.0, 76.0),          # should be <= 76 kPa
]

# Callback
n_iter_slsqp = 0


def callback_slsqp(xk):
    global n_iter_slsqp
    n_iter_slsqp += 1
    print(f"Iter {n_iter_slsqp} | Evaluations {ev.n_evaluations}")


# Options
method = 'SLSQP'
options = {
    'maxiter': 100,
    'ftol': 1e-2,
    'disp': True,
    'eps': 0.5
}

# Run optimization
start_time = time.time()
res_slsqp = minimize(
    ev.objective,
    x0,
    method=method,
    bounds=bounds,
    constraints=constraints_scipy,
    options=options,
    callback=callback_slsqp
)
time_slsqp = time.time() - start_time



OPTIMIZATION 1: SLSQP (SciPy - Local Optimizer)
Iter 1 | Evaluations 9
Iter 2 | Evaluations 15
Iter 3 | Evaluations 21
Iter 4 | Evaluations 27
Iter 5 | Evaluations 32
Optimization terminated successfully    (Exit mode 0)
            Current function value: -663.8537091855615
            Iterations: 5
            Function evaluations: 30
            Gradient evaluations: 5


In [29]:
# Results
print("\n" + "=" * 60)
print("SLSQP RESULTS:")
print("=" * 60)
print(f"Converged: {res_slsqp.success}")
print(f"Message: {res_slsqp.message}")
print(f"\nObjective Value:")
print(f"  Sales Price = {-res_slsqp.fun:.2f}")

print(f"\nOptimal Solution:")
print(f"  V_123102_Temp = {res_slsqp.x[0]:.2f}")
print(f"  T_123701_Reboiler = {res_slsqp.x[1]:.2f}")
print(f"  T_123702_Condenser = {res_slsqp.x[2]:.2f}")
print(f"  T_123702_Reboiler = {res_slsqp.x[3]:.2f}")
print(f"  T_123703_Reboiler = {res_slsqp.x[4]:.2f}")


print(f"\nPerformance:")
print(f"  Iterations: {n_iter_slsqp}")
print(f"  DWSIM evaluations: {ev.n_evaluations}")
print(f"  Optimization time: {time_slsqp:.2f} seconds")

# Check constraints
report = check_constraints_scipy(res_slsqp.x, constraints_scipy)
print("\nConstraint Check:")
for r in report:
    status = "✓ OK" if r['ok'] else "✗ VIOLATED"
    print(f"  Constraint {r['id']}: {status}")
    print(f"    value = {r['value'][0]:.4f}, bounds = [{r['lb'][0]:.2f}, {r['ub'][0]:.2f}]")



SLSQP RESULTS:
Converged: True
Message: Optimization terminated successfully

Objective Value:
  Sales Price = 663.85

Optimal Solution:
  V_123102_Temp = -33.00
  T_123701_Reboiler = 63.53
  T_123702_Condenser = 4.31
  T_123702_Reboiler = 15.96
  T_123703_Reboiler = 48.00

Performance:
  Iterations: 5
  DWSIM evaluations: 32
  Optimization time: 229.80 seconds

Constraint Check:
  Constraint 1: ✓ OK
    value = 83.7441, bounds = [80.00, inf]
  Constraint 2: ✓ OK
    value = 11.9999, bounds = [0.00, 12.00]
  Constraint 3: ✓ OK
    value = 0.0056, bounds = [0.00, 2.00]
  Constraint 4: ✗ VIOLATED
    value = 76.0021, bounds = [0.00, 76.00]


## Run Optimal Solution

In [30]:
Optimal = res_slsqp.x

In [31]:
print(Optimal)

[-33.          63.52815876   4.31232576  15.95793763  47.9998889 ]


In [32]:
print("=" * 60)
print("Optimal Solution")
print("=" * 60)

(f_obj, g1, g2, g3, g4,
 T_123701_Stage1_Temperature,
 T_123701_Stage11_Temperature,
 T_123702_Stage1_Temperature,
 T_123702_Stage21_Temperature,
 T_123703_Stage1_Temperature,
 T_123703_Stage6_Temperature) = run_dwsim_simulation(Optimal)

print(f"\nTest Input:")
print(f"  V_123102_Temp = {Optimal[0]:.2f}")
print(f"  T_123701_Reboiler = {Optimal[1]:.2f}")
print(f"  T_123702_Condenser = {Optimal[2]:.2f}")
print(f"  T_123702_Reboiler = {Optimal[3]:.2f}")
print(f"  T_123703_Reboiler = {Optimal[4]:.2f}")

print(f"\nTest Output:")
print(f"  Objective (Sales Price) = {-f_obj:.2f} US$")
print(f"  g1 (GV.C1 [%]) = {g1:.2f} (should be >= 80 %)")
print(f"  g2 (GLP.C2 [%]) = {g2:.2f} (should be <=12 %)")
print(f"  g3 (GLP.C5 [%]) = {g3:.2f} (should be <=2 %)")
print(f"  g4 (C5+.PVR [kPa]) = {g4:.2f} (should be <= 76 kPa)")

print(f"\nTemperatures Output:")
print(f"  T 123701 Stage 1 Temperature = {T_123701_Stage1_Temperature:.2f} ºC")
print(f"  T 123701 Stage 11 Temperature = {T_123701_Stage11_Temperature:.2f} ºC")
print(f"  T 123702 Stage 1 Temperature = {T_123702_Stage1_Temperature:.2f} ºC")
print(f"  T 123702 Stage 21 Temperature = {T_123702_Stage21_Temperature:.2f} ºC")
print(f"  T 123703 Stage 1 Temperature = {T_123703_Stage1_Temperature:.2f} ºC")
print(f"  T 123703 Stage 6 Temperature = {T_123703_Stage6_Temperature:.2f} ºC")

Optimal Solution

Test Input:
  V_123102_Temp = -33.00
  T_123701_Reboiler = 63.53
  T_123702_Condenser = 4.31
  T_123702_Reboiler = 15.96
  T_123703_Reboiler = 48.00

Test Output:
  Objective (Sales Price) = 663.85 US$
  g1 (GV.C1 [%]) = 83.74 (should be >= 80 %)
  g2 (GLP.C2 [%]) = 12.00 (should be <=12 %)
  g3 (GLP.C5 [%]) = 0.01 (should be <=2 %)
  g4 (C5+.PVR [kPa]) = 76.00 (should be <= 76 kPa)

Temperatures Output:
  T 123701 Stage 1 Temperature = 15.73 ºC
  T 123701 Stage 11 Temperature = 66.29 ºC
  T 123702 Stage 1 Temperature = 34.29 ºC
  T 123702 Stage 21 Temperature = 140.11 ºC
  T 123703 Stage 1 Temperature = 107.44 ºC
  T 123703 Stage 6 Temperature = 197.24 ºC


## ROBUST OPTIMIZATION

In [35]:
import os
FIGDIR = 'operability/optimization_figures'
os.makedirs(FIGDIR, exist_ok=True)

In [36]:
# ============================================================
# CONFIGURATION
# ============================================================

# Revenue from the original SLSQP optimal (constraint-boundary solution)
REV_OPTIMAL = -f_obj  # $/h

# Constraint limits (ANP regulations)
LIM_G1 = 80.0   # SG_C1 >= 80 mol%
LIM_G2 = 12.0   # LPG_C2 <= 12 mol%
LIM_G3 = 2.0    # LPG_C5 <= 2 mol%
LIM_G4 = 76.0   # NG_RVP <= 76 kPa



In [37]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def compute_margins(ev, x):
    """Compute normalised constraint margins (0 = at boundary, 1 = far away)."""
    g1 = ev.g1(x)
    g2 = ev.g2(x)
    g3 = ev.g3(x)
    g4 = ev.g4(x)

    margin_g1 = (g1 - LIM_G1) / LIM_G1        # >= 0 when feasible
    margin_g2 = (LIM_G2 - g2) / LIM_G2         # >= 0 when feasible
    margin_g3 = (LIM_G3 - g3) / LIM_G3         # >= 0 when feasible
    margin_g4 = (LIM_G4 - g4) / LIM_G4         # >= 0 when feasible

    return {
        "SG_C1":  {"value": g1, "margin": margin_g1},
        "LPG_C2": {"value": g2, "margin": margin_g2},
        "LPG_C5": {"value": g3, "margin": margin_g3},
        "NG_RVP": {"value": g4, "margin": margin_g4},
    }


def print_solution(label, x, ev, elapsed=None):
    """Pretty-print an optimization solution."""
    rev = -ev.objective(x)
    margins = compute_margins(ev, x)
    min_margin = min(m["margin"] for m in margins.values())

    print(f"\n{'=' * 60}")
    print(f"  {label}")
    print(f"{'=' * 60}")
    print(f"  Revenue = {rev:.2f} $/h")
    print(f"  Min normalised margin = {min_margin:.4f} ({min_margin*100:.2f}%)")
    if elapsed:
        print(f"  Elapsed = {elapsed:.1f} s")

    print(f"\n  Decision variables:")
    var_names = ["V_02_Temp", "T_01_Reb", "T_02_RR", "T_02_Reb", "T_03_Reb"]
    for name, val in zip(var_names, x):
        print(f"    {name:<14} = {val:>10.4f}")

    print(f"\n  Constraints:")
    print(f"    {'Constraint':<12} {'Value':>10} {'Limit':>10} {'Margin':>10} {'Status'}")
    print(f"    {'-'*56}")
    for name, info in margins.items():
        status = "✓" if info["margin"] >= 0 else "✗ VIOLATED"
        margin_pct = f"{info['margin']*100:+.2f}%"
        lim = {"SG_C1": f">= {LIM_G1}", "LPG_C2": f"<= {LIM_G2}",
               "LPG_C5": f"<= {LIM_G3}", "NG_RVP": f"<= {LIM_G4}"}[name]
        print(f"    {name:<12} {info['value']:>10.4f} {lim:>10} {margin_pct:>10} {status}")

    return rev, min_margin




In [38]:
# ============================================================
# APPROACH 1: BACK-OFF (tighten constraint limits)
# ============================================================

def run_backoff_optimization(ev, x0, bounds, margin_pct=0.05):
    """
    Simple back-off: tighten each constraint by a fixed percentage.

    margin_pct = 0.05 means 5% safety margin:
      LPG_C2 <= 12 * 0.95 = 11.4
      LPG_C5 <= 2  * 0.95 = 1.9
      NG_RVP <= 76 * 0.95 = 72.2
      SG_C1  >= 80 * 1.05 = 84.0
    """
    print(f"\n\n{'#' * 60}")
    print(f"APPROACH 1: BACK-OFF ({margin_pct*100:.0f}% safety margin)")
    print(f"{'#' * 60}")

    constraints_backoff = [
        NonlinearConstraint(ev.g1, LIM_G1 * (1 + margin_pct), np.inf),
        NonlinearConstraint(ev.g2, 0.0, LIM_G2 * (1 - margin_pct)),
        NonlinearConstraint(ev.g3, 0.0, LIM_G3 * (1 - margin_pct)),
        NonlinearConstraint(ev.g4, 0.0, LIM_G4 * (1 - margin_pct)),
    ]

    ev.reset_counter()
    ev.cache.clear()

    start = time.time()
    res = minimize(
        ev.objective,
        x0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints_backoff,
        options={'maxiter': 100, 'ftol': 1e-2, 'eps': 0.5}
    )
    elapsed = time.time() - start

    print(f"\n  Optimizer status: {res.message}")
    print(f"  DWSIM evaluations: {ev.n_evaluations}")
    rev, min_margin = print_solution(
        f"Back-off {margin_pct*100:.0f}% — Result", res.x, ev, elapsed)

    return res, rev, min_margin


# ============================================================
# APPROACH 2: PENALISED OBJECTIVE (Revenue × Robustness trade-off)
# ============================================================

def run_penalised_optimization(ev, x0, bounds, alpha=0.1):
    """
    Penalise the objective by inverse distance to the closest constraint.

    alpha controls the weight:
      alpha = 0    → pure revenue (recovers original SLSQP)
      alpha = 0.1  → mild robustness preference
      alpha = 1.0  → strong robustness preference
    """
    print(f"\n\n{'#' * 60}")
    print(f"APPROACH 2: PENALISED OBJECTIVE (alpha = {alpha})")
    print(f"{'#' * 60}")

    def penalised_objective(x):
        rev = ev.objective(x)  # negative (minimisation)

        g2 = ev.g2(x)
        g3 = ev.g3(x)
        g4 = ev.g4(x)

        margin_g2 = (LIM_G2 - g2) / LIM_G2
        margin_g3 = (LIM_G3 - g3) / LIM_G3
        margin_g4 = (LIM_G4 - g4) / LIM_G4

        min_margin = min(margin_g2, margin_g3, margin_g4)

        # Barrier-like penalty: grows as margin → 0
        # Clamped to avoid division by zero
        safe_margin = max(min_margin, 0.001)
        penalty = alpha * abs(rev) * (1.0 / safe_margin - 1.0)

        return rev + penalty

    constraints_original = [
        NonlinearConstraint(ev.g1, LIM_G1, np.inf),
        NonlinearConstraint(ev.g2, 0.0, LIM_G2),
        NonlinearConstraint(ev.g3, 0.0, LIM_G3),
        NonlinearConstraint(ev.g4, 0.0, LIM_G4),
    ]

    ev.reset_counter()
    ev.cache.clear()

    start = time.time()
    res = minimize(
        penalised_objective,
        x0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints_original,
        options={'maxiter': 100, 'ftol': 1e-2, 'eps': 0.5}
    )
    elapsed = time.time() - start

    print(f"\n  Optimizer status: {res.message}")
    print(f"  DWSIM evaluations: {ev.n_evaluations}")
    rev, min_margin = print_solution(
        f"Penalised (alpha={alpha}) — Result", res.x, ev, elapsed)

    return res, rev, min_margin


# ============================================================
# APPROACH 3: TWO-STAGE (Maximise min-margin subject to min revenue)
# ============================================================

def run_twostage_optimization(ev, x0, bounds, rev_min_fraction=0.95):
    """
    Stage 1: Already done (SLSQP → 663.85 $/h)
    Stage 2: Maximise the minimum normalised constraint margin,
             subject to Revenue >= rev_min_fraction * REV_OPTIMAL.

    This finds the point FARTHEST from all constraint boundaries
    while still achieving acceptable revenue.
    """
    rev_min = REV_OPTIMAL * rev_min_fraction

    print(f"\n\n{'#' * 60}")
    print(f"APPROACH 3: TWO-STAGE (min revenue = {rev_min:.2f} $/h, "
          f"{rev_min_fraction*100:.0f}% of optimal)")
    print(f"{'#' * 60}")

    def min_margin_objective(x):
        """Minimise the negative of the smallest margin → maximise smallest margin."""
        g2 = ev.g2(x)
        g3 = ev.g3(x)
        g4 = ev.g4(x)

        margin_g2 = (LIM_G2 - g2) / LIM_G2
        margin_g3 = (LIM_G3 - g3) / LIM_G3
        margin_g4 = (LIM_G4 - g4) / LIM_G4

        return -min(margin_g2, margin_g3, margin_g4)

    def revenue_constraint(x):
        """Revenue must be >= rev_min."""
        return -ev.objective(x)  # objective returns negative → negate

    constraints_stage2 = [
        NonlinearConstraint(ev.g1, LIM_G1, np.inf),
        NonlinearConstraint(ev.g2, 0.0, LIM_G2),
        NonlinearConstraint(ev.g3, 0.0, LIM_G3),
        NonlinearConstraint(ev.g4, 0.0, LIM_G4),
        NonlinearConstraint(revenue_constraint, rev_min, np.inf),
    ]

    ev.reset_counter()
    ev.cache.clear()

    start = time.time()
    res = minimize(
        min_margin_objective,
        x0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints_stage2,
        options={'maxiter': 100, 'ftol': 1e-3, 'eps': 0.5}
    )
    elapsed = time.time() - start

    print(f"\n  Optimizer status: {res.message}")
    print(f"  DWSIM evaluations: {ev.n_evaluations}")
    rev, min_margin = print_solution(
        f"Two-stage ({rev_min_fraction*100:.0f}% revenue) — Result",
        res.x, ev, elapsed)

    return res, rev, min_margin


# ============================================================
# PARETO CURVE: Revenue vs Minimum Margin
# ============================================================

def run_pareto_analysis(ev, x0, bounds,
                        fractions=None):
    """
    Sweep revenue targets from 100% down to 80% of optimal,
    at each level maximising the minimum constraint margin.
    Builds a Pareto frontier: Revenue vs Robustness.
    """
    if fractions is None:
        # Max joint margin is ~4.5% — finer sweep near the top matters more
        fractions = [1.000, 0.998, 0.995, 0.993, 0.990, 0.985,
                     0.980, 0.975, 0.970, 0.960, 0.950, 0.930]

    print(f"\n\n{'#' * 60}")
    print(f"PARETO ANALYSIS: Revenue vs Minimum Constraint Margin")
    print(f"{'#' * 60}")

    pareto_results = []
    current_x0 = x0.copy()

    for frac in fractions:
        rev_min = REV_OPTIMAL * frac

        def min_margin_objective(x):
            g2 = ev.g2(x)
            g3 = ev.g3(x)
            g4 = ev.g4(x)
            margin_g2 = (LIM_G2 - g2) / LIM_G2
            margin_g3 = (LIM_G3 - g3) / LIM_G3
            margin_g4 = (LIM_G4 - g4) / LIM_G4
            return -min(margin_g2, margin_g3, margin_g4)

        def revenue_constraint(x):
            return -ev.objective(x)

        constraints_pareto = [
            NonlinearConstraint(ev.g1, LIM_G1, np.inf),
            NonlinearConstraint(ev.g2, 0.0, LIM_G2),
            NonlinearConstraint(ev.g3, 0.0, LIM_G3),
            NonlinearConstraint(ev.g4, 0.0, LIM_G4),
            NonlinearConstraint(revenue_constraint, rev_min, np.inf),
        ]

        ev.reset_counter()
        ev.cache.clear()

        try:
            res = minimize(
                min_margin_objective,
                current_x0,
                method='SLSQP',
                bounds=bounds,
                constraints=constraints_pareto,
                options={'maxiter': 100, 'ftol': 1e-3, 'eps': 0.5}
            )

            rev = -ev.objective(res.x)
            margins = compute_margins(ev, res.x)
            min_margin = min(m["margin"] for m in margins.values())

            # Use this solution as starting point for next (warm-start)
            current_x0 = res.x.copy()

            status = "✓" if res.success else "~"
            print(f"  {status} Target {frac*100:5.1f}% | Rev = {rev:7.2f} $/h | "
                  f"Min margin = {min_margin*100:6.2f}% | Evals = {ev.n_evaluations}")

            pareto_results.append({
                "fraction": frac,
                "rev_target": rev_min,
                "rev_actual": rev,
                "min_margin": min_margin,
                "x": res.x.copy(),
                "margins": margins,
                "success": res.success,
            })

        except Exception as e:
            print(f"  ✗ Target {frac*100:5.1f}% | FAILED: {e}")

    return pareto_results


def plot_pareto(pareto_results):
    """Plot the Pareto frontier: Revenue vs Minimum Constraint Margin."""

    revs = [r["rev_actual"] for r in pareto_results]
    margins = [r["min_margin"] * 100 for r in pareto_results]

    fig, ax = plt.subplots(figsize=(8, 5))

    # Pareto frontier
    ax.plot(margins, revs, "o-", color="#2c3e50", markersize=8, linewidth=2,
            label="Pareto frontier", zorder=5)

    # Annotate key points
    for r in pareto_results:
        if r["fraction"] in [1.00, 0.95, 0.90, 0.85]:
            ax.annotate(
                f'{r["rev_actual"]:.0f} $/h\n({r["min_margin"]*100:.1f}%)',
                xy=(r["min_margin"] * 100, r["rev_actual"]),
                xytext=(15, -10), textcoords="offset points",
                fontsize=8, ha="left",
                arrowprops=dict(arrowstyle="->", color="gray", lw=0.8))

    # Original SLSQP optimal (on boundary)
    ax.scatter([0], [REV_OPTIMAL], c="#d62728", s=120, marker="*",
               zorder=10, edgecolors="black", linewidths=0.5,
               label=f"SLSQP optimal ({REV_OPTIMAL:.0f} $/h, 0% margin)")

    # Reference lines
    ax.axhline(REV_OPTIMAL, color="gray", ls=":", lw=0.8, alpha=0.5)
    ax.axvline(5, color="gray", ls=":", lw=0.8, alpha=0.5)
    ax.text(5.2, ax.get_ylim()[0] + 2, "5% margin", fontsize=7, color="gray")

    ax.set_xlabel("Minimum constraint margin (%)", fontsize=11)
    ax.set_ylabel("Revenue ($/h)", fontsize=11)
    ax.set_title("Pareto Frontier: Revenue vs Operational Robustness",
                 fontsize=12, fontweight="bold")
    ax.legend(loc="lower right", framealpha=0.9)

    # Shade the "recommended zone"
    from matplotlib.patches import Rectangle
    rec = Rectangle((3, ax.get_ylim()[0]), 10, ax.get_ylim()[1] - ax.get_ylim()[0],
                     facecolor="#2ca02c", alpha=0.05, zorder=0)
    ax.add_patch(rec)
    ax.text(8, ax.get_ylim()[1] - 5, "Recommended\noperating zone",
            fontsize=8, color="#2ca02c", ha="center", style="italic")

    plt.tight_layout()
    plt.savefig(f"{FIGDIR}/pareto_revenue_vs_margin.png")
    plt.savefig(f"{FIGDIR}/pareto_revenue_vs_margin.pdf")
    plt.close()
    print(f"\n  ✓ Pareto figure saved to {FIGDIR}/")

    return fig


def plot_margin_breakdown(pareto_results):
    """Stacked bar chart showing individual constraint margins at each Pareto point."""

    fracs = [r["fraction"] * 100 for r in pareto_results]
    labels = [f"{f:.0f}%" for f in fracs]

    m_g2 = [r["margins"]["LPG_C2"]["margin"] * 100 for r in pareto_results]
    m_g3 = [r["margins"]["LPG_C5"]["margin"] * 100 for r in pareto_results]
    m_g4 = [r["margins"]["NG_RVP"]["margin"] * 100 for r in pareto_results]

    x_pos = np.arange(len(pareto_results))
    width = 0.25

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x_pos - width, m_g2, width, label="LPG_C2 margin", color="#3498db", alpha=0.85)
    ax.bar(x_pos,         m_g3, width, label="LPG_C5 margin", color="#9b59b6", alpha=0.85)
    ax.bar(x_pos + width, m_g4, width, label="NG_RVP margin", color="#e74c3c", alpha=0.85)

    # Revenue annotation on secondary axis
    ax2 = ax.twinx()
    ax2.plot(x_pos, [r["rev_actual"] for r in pareto_results],
             "ko--", markersize=5, linewidth=1, alpha=0.6, label="Revenue")
    ax2.set_ylabel("Revenue ($/h)", fontsize=10)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_xlabel("Revenue target (% of optimal)", fontsize=10)
    ax.set_ylabel("Constraint margin (%)", fontsize=10)
    ax.set_title("Constraint Margin Breakdown along Pareto Frontier",
                 fontsize=12, fontweight="bold")
    ax.legend(loc="upper left", fontsize=8, framealpha=0.9)
    ax2.legend(loc="upper right", fontsize=8, framealpha=0.9)

    plt.tight_layout()
    plt.savefig(f"{FIGDIR}/pareto_margin_breakdown.png")
    plt.savefig(f"{FIGDIR}/pareto_margin_breakdown.pdf")
    plt.close()
    print(f"  ✓ Margin breakdown figure saved to {FIGDIR}/")

    return fig

In [39]:
# ============================================================
# RUN ALL APPROACHES
# ============================================================
# Assumes ev, x0, bounds are already defined in the notebook

print("\n" + "=" * 60)
print("ROBUST OPTIMIZATION ANALYSIS")
print("=" * 60)

# ── 1. Original SLSQP (for reference) ──
print("\n\n" + "=" * 60)
print("REFERENCE: Original SLSQP (boundary-hugging)")
print("=" * 60)

ev.reset_counter()
ev.cache.clear()

constraints_original = [
    NonlinearConstraint(ev.g1, LIM_G1, np.inf),
    NonlinearConstraint(ev.g2, 0.0, LIM_G2),
    NonlinearConstraint(ev.g3, 0.0, LIM_G3),
    NonlinearConstraint(ev.g4, 0.0, LIM_G4),
]

start = time.time()
res_original = minimize(
    ev.objective, x0, method='SLSQP', bounds=bounds,
    constraints=constraints_original,
    options={'maxiter': 100, 'ftol': 1e-2, 'eps': 0.5}
)
time_original = time.time() - start
rev_orig, margin_orig = print_solution(
    "Original SLSQP", res_original.x, ev, time_original)




ROBUST OPTIMIZATION ANALYSIS


REFERENCE: Original SLSQP (boundary-hugging)

  Original SLSQP
  Revenue = 663.85 $/h
  Min normalised margin = -0.0000 (-0.00%)
  Elapsed = 218.1 s

  Decision variables:
    V_02_Temp      =   -33.0000
    T_01_Reb       =    63.5282
    T_02_RR        =     4.3119
    T_02_Reb       =    15.9579
    T_03_Reb       =    48.0000

  Constraints:
    Constraint        Value      Limit     Margin Status
    --------------------------------------------------------
    SG_C1           83.7441    >= 80.0     +4.68% ✓
    LPG_C2          11.9999    <= 12.0     +0.00% ✓
    LPG_C5           0.0056     <= 2.0    +99.72% ✓
    NG_RVP          76.0021    <= 76.0     -0.00% ✗ VIOLATED


In [40]:
# ── 2. Back-off approaches ──
res_bo1, rev_bo1, margin_bo1 = run_backoff_optimization(
    ev, res_original.x, bounds, margin_pct=0.01)
res_bo2, rev_bo2, margin_bo2 = run_backoff_optimization(
    ev, res_original.x, bounds, margin_pct=0.02)
res_bo3, rev_bo3, margin_bo3 = run_backoff_optimization(
    ev, res_original.x, bounds, margin_pct=0.03)



############################################################
APPROACH 1: BACK-OFF (1% safety margin)
############################################################

  Optimizer status: Optimization terminated successfully
  DWSIM evaluations: 14

  Back-off 1% — Result
  Revenue = 663.76 $/h
  Min normalised margin = 0.0100 (1.00%)
  Elapsed = 87.3 s

  Decision variables:
    V_02_Temp      =   -33.0000
    T_01_Reb       =    63.3824
    T_02_RR        =     4.3146
    T_02_Reb       =    15.9062
    T_03_Reb       =    47.9648

  Constraints:
    Constraint        Value      Limit     Margin Status
    --------------------------------------------------------
    SG_C1           83.7368    >= 80.0     +4.67% ✓
    LPG_C2          11.8795    <= 12.0     +1.00% ✓
    LPG_C5           0.0057     <= 2.0    +99.72% ✓
    NG_RVP          75.2309    <= 76.0     +1.01% ✓


############################################################
APPROACH 1: BACK-OFF (2% safety margin)
###################

In [ ]:
# ── 3. Penalised objective ──
# With max joint margin of ~4.5%, use gentle alpha values
res_pen01, rev_pen01, margin_pen01 = run_penalised_optimization(
    ev, res_original.x, bounds, alpha=0.01)
res_pen05, rev_pen05, margin_pen05 = run_penalised_optimization(
    ev, res_original.x, bounds, alpha=0.05)
res_pen10, rev_pen10, margin_pen10 = run_penalised_optimization(
    ev, res_original.x, bounds, alpha=0.10)



############################################################
APPROACH 2: PENALISED OBJECTIVE (alpha = 0.01)
############################################################


In [ ]:
# ── 4. Two-stage optimizations ──
# Max joint margin is ~4.5%, so even 99% revenue is tight
res_ts99, rev_ts99, margin_ts99 = run_twostage_optimization(
    ev, res_original.x, bounds, rev_min_fraction=0.995)
res_ts98, rev_ts98, margin_ts98 = run_twostage_optimization(
    ev, res_original.x, bounds, rev_min_fraction=0.99)
res_ts95, rev_ts95, margin_ts95 = run_twostage_optimization(
    ev, res_original.x, bounds, rev_min_fraction=0.95)

In [ ]:
# ── 5. Full Pareto analysis ──
pareto_results = run_pareto_analysis(ev, res_original.x, bounds)

# ── 6. Plot results ──
if pareto_results:
    plot_pareto(pareto_results)
    plot_margin_breakdown(pareto_results)

In [ ]:
# ── 7. Summary comparison table ──
print(f"\n\n{'=' * 80}")
print("SUMMARY: All approaches compared")
print(f"{'=' * 80}")
print(f"{'Approach':<35} {'Revenue':>10} {'Min Margin':>12} {'Trade-off':>10}")
print(f"{'-'*70}")
print(f"{'Original SLSQP':<35} {rev_orig:>10.2f} {margin_orig*100:>11.2f}% {'---':>10}")
print(f"{'Back-off 1%':<35} {rev_bo1:>10.2f} {margin_bo1*100:>11.2f}% "
      f"{rev_orig-rev_bo1:>+9.2f}")
print(f"{'Back-off 2%':<35} {rev_bo2:>10.2f} {margin_bo2*100:>11.2f}% "
      f"{rev_orig-rev_bo2:>+9.2f}")
print(f"{'Back-off 3%':<35} {rev_bo3:>10.2f} {margin_bo3*100:>11.2f}% "
      f"{rev_orig-rev_bo3:>+9.2f}")
print(f"{'Penalised (alpha=0.01)':<35} {rev_pen01:>10.2f} {margin_pen01*100:>11.2f}% "
      f"{rev_orig-rev_pen01:>+9.2f}")
print(f"{'Penalised (alpha=0.05)':<35} {rev_pen05:>10.2f} {margin_pen05*100:>11.2f}% "
      f"{rev_orig-rev_pen05:>+9.2f}")
print(f"{'Penalised (alpha=0.10)':<35} {rev_pen10:>10.2f} {margin_pen10*100:>11.2f}% "
      f"{rev_orig-rev_pen10:>+9.2f}")
print(f"{'Two-stage (99.5% rev.)':<35} {rev_ts99:>10.2f} {margin_ts99*100:>11.2f}% "
      f"{rev_orig-rev_ts99:>+9.2f}")
print(f"{'Two-stage (99.0% rev.)':<35} {rev_ts98:>10.2f} {margin_ts98*100:>11.2f}% "
      f"{rev_orig-rev_ts98:>+9.2f}")
print(f"{'Two-stage (95.0% rev.)':<35} {rev_ts95:>10.2f} {margin_ts95*100:>11.2f}% "
      f"{rev_orig-rev_ts95:>+9.2f}")

print(f"\n{'=' * 80}")
print("RECOMMENDATION:")
print(f"{'=' * 80}")
print("""
  DATA INSIGHT: The feasible region is EXTREMELY tight.
  The bottleneck is SG_C1 >= 80 (methane in Sales Gas), which limits
  the joint normalised margin to a MAXIMUM of ~4.5%.
  No feasible point exists with all margins >= 5%.

  This means:
  - Back-off > 3% may be infeasible or nearly so
  - The two-stage approach is recommended because it finds the point
    with MAXIMUM distance to all boundaries within what's achievable
  - The Pareto curve quantifies the exact Revenue vs Margin trade-off
  - Sacrificing ~1-2% of revenue buys most of the available margin

  For the paper, this is a strong result: the low OI (14.1%) is not
  just about constraint satisfaction rate — even WITHIN the feasible
  region, the margins are razor-thin, reinforcing the fragility finding.
""")